# 📈 Análisis Cuantitativo Automatizado del S&P 500 — V5 "Las 3 mejoras"

Misma estructura que **V4** (Pasos 1–7 + resumen del Top 5) con las tres mejoras que el backtest de 20 años validó:

| # | Mejora | Dónde se aplica | Qué cambia |
|---|---|---|---|
| 1 | **Tasa libre de riesgo dinámica** | Fase 1.3 y todas las fases que usan Sharpe/CAPM | Se usa la tasa actual del T-Bill de 13 semanas (`^IRX`) en lugar de 4 % fija. |
| 2 | **Filtro de mercado SMA 200** | Fase 3.3 y Recomendación final | Si SPY cierra por debajo de su media de 200 días, la recomendación es **efectivo / T-Bills**. |
| 3 | **Universo histórico (point-in-time)** | Paso 8 (backtest) | En cada fecha del backtest solo se eligen acciones que **en ese momento** eran miembros del S&P 500. Hoy el universo histórico es la lista actual. |

**Paso 8** (opcional, ~10–15 min en Colab) corre el backtest walk-forward de 20 años con las tres mejoras, Monte Carlo y la lista de todas las operaciones.

> ⚠️ Proyecto educativo. Los resultados se basan en datos históricos y **no constituyen recomendación de inversión**.

**Colab:** `Entorno de ejecución → Ejecutar todas`. La primera celda instala las dependencias.

In [ ]:
# Dependencias (Colab ya trae pandas, numpy, scipy, matplotlib, seaborn y statsmodels)
!pip install -q yfinance lxml

# Paso 1: Filtro Inicial de Acciones

### Fase 1.1: Descarga y Filtrado de Acciones por Volumen Promedio

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
import requests
from io import StringIO

# Ignorar warnings para una salida más limpia
warnings.filterwarnings('ignore', category=FutureWarning)

print("Obteniendo la lista de tickers del S&P 500 desde Wikipedia...")
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
# Usar requests con un User-Agent personalizado para evitar el error 403
header = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
response = requests.get(url, headers=header)
# pandas >= 2.1 requiere envolver el HTML en StringIO (pasar el texto directo falla en pandas 3)
tabla_sp500 = pd.read_html(StringIO(response.text))[0]
# Reemplazar puntos por guiones para la compatibilidad con yfinance (ej. BRK.B -> BRK-B)
tickers_muestra = tabla_sp500['Symbol'].str.replace('.', '-', regex=False).tolist()

# Diccionario para almacenar las acciones que pasen el filtro
acciones_validas = {}

print(f"Descargando datos para {len(tickers_muestra)} acciones usando yfinance (descarga en bloque para mayor velocidad)...")

# Descargar los últimos 5 años en bloque, incluyendo SPY como referencia de mercado.
# auto_adjust=True: 'Close' ya viene ajustado por splits y dividendos (retorno total).
datos_bloque = yf.download(tickers_muestra + ['SPY'], period="5y", progress=False, auto_adjust=True)


def extraer_ticker(ticker):
    """Devuelve el OHLCV de un ticker como DataFrame con columna 'Date'."""
    # Al descargar múltiples tickers, yfinance devuelve un MultiIndex (Precio, Ticker)
    if isinstance(datos_bloque.columns, pd.MultiIndex):
        df = datos_bloque.xs(ticker, axis=1, level=1, drop_level=True).copy()
    else:
        df = datos_bloque.copy()
    df = df.dropna(subset=['Close']).reset_index()
    if 'index' in df.columns and 'Date' not in df.columns:
        df = df.rename(columns={'index': 'Date'})
    return df


# Filtrar acciones que tengan un volumen promedio mayor a 1,000,000
for ticker in tickers_muestra:
    try:
        datos_temp = extraer_ticker(ticker)
        if not datos_temp.empty and 'Volume' in datos_temp.columns:
            if datos_temp['Volume'].mean() > 1_000_000:
                # Ticker en minúsculas para mantener la compatibilidad con el resto del código
                acciones_validas[ticker.lower()] = datos_temp
    except Exception as error:
        print(f"Error procesando {ticker}: {error}")

# SPY se guarda aparte: es el mercado, no una acción candidata
datos_spy = extraer_ticker('SPY')

print(f"Se encontraron {len(acciones_validas)} acciones válidas de {len(tickers_muestra)} analizadas.")
print("Primeras 30 acciones válidas encontradas:", list(acciones_validas.keys())[:30])

### Fase 1.2: Consolidación de Precios de Cierre y Retornos Diarios

In [ ]:
# Consolidar los precios de cierre ajustados de las acciones válidas
lista_precios = []
for ticker, df_accion in acciones_validas.items():
    df_temp = df_accion[['Date', 'Close']].copy()
    df_temp['Date'] = pd.to_datetime(df_temp['Date'])
    df_temp = df_temp.rename(columns={'Close': ticker})
    lista_precios.append(df_temp.set_index('Date'))

# Unión 'outer' + filtro de cobertura. Un join='inner' recortaría TODO el historial
# a la fecha de inicio de la acción más reciente (p. ej. una IPO de hace un mes).
df_precios_unidos = pd.concat(lista_precios, axis=1, join='outer').sort_index()
min_obs = int(0.95 * len(df_precios_unidos))
cobertura = df_precios_unidos.notna().sum()
acciones_excluidas = cobertura[cobertura < min_obs].index.tolist()
df_precios_unidos = df_precios_unidos.drop(columns=acciones_excluidas)

# Agregar SPY (ya descargado en la celda anterior) como columna 'Market'
spy_close = datos_spy.set_index(pd.to_datetime(datos_spy['Date']))['Close'].rename('Market')
df_precios_unidos = df_precios_unidos.join(spy_close, how='inner').dropna()

# Calcular retornos diarios
retornos_diarios = df_precios_unidos.pct_change(fill_method=None).dropna()
# Retornos solo de las acciones (sin el mercado); se usa en las fases siguientes
retornos_activos = retornos_diarios.drop(columns=['Market'])

print(f"Acciones excluidas por historial corto (<95% de las fechas): {len(acciones_excluidas)} -> {acciones_excluidas}")
print(f"Rango de datos: {retornos_diarios.index.min().date()} a {retornos_diarios.index.max().date()} "
      f"({len(retornos_diarios)} días, {retornos_activos.shape[1]} acciones)")
print("Primeras filas de los retornos diarios (incluyendo SPY como 'Market'):")
display(retornos_diarios.head())

### Fase 1.3: Tasa Libre de Riesgo Dinámica (Mejora 1)
En V4 la tasa libre de riesgo era un 4 % fijo. Aquí se descarga el rendimiento del **T-Bill de 13 semanas** (`^IRX`, en % anual) y se usa su último valor en el CAPM, el Sharpe y la optimización. En el backtest (Paso 8) se usa la tasa vigente en **cada** fecha.

In [ ]:
# MEJORA 1: tasa libre de riesgo del T-Bill de 13 semanas (^IRX viene en porcentaje anual)
datos_irx = yf.download('^IRX', period='5y', progress=False, auto_adjust=True)['Close'].squeeze().dropna()
tasa_libre_riesgo_actual = float(datos_irx.iloc[-1]) / 100

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3))
plt.plot(datos_irx.index, datos_irx / 100, color='teal')
plt.axhline(0.04, color='gray', ls='--', label='4 % fijo usado en V4')
plt.title('Tasa del T-Bill de 13 semanas (^IRX)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
plt.legend(); plt.grid(True); plt.show()

print(f"Tasa libre de riesgo actual (T-Bill 13 semanas): {tasa_libre_riesgo_actual:.2%}  (V4 usaba 4.00 % fijo)")

# Paso 2: Análisis Fundamental y Modelos Financieros

### Fase 2.1: Cálculo del CAPM y Beta de Hamada

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Definir variables financieras de tasa libre de riesgo y retorno de mercado
tasa_libre_riesgo_anual = tasa_libre_riesgo_actual  # MEJORA 1: T-Bill actual (V4: 0.04 fijo)
tasa_impuestos = 0.21
retorno_mercado_anual = retornos_diarios['Market'].mean() * 252
varianza_mercado = retornos_diarios['Market'].var() * 252
prima_mercado = retorno_mercado_anual - tasa_libre_riesgo_anual

metricas_acciones = []

print("Calculando métricas y consultando Debt-to-Equity en Yahoo Finance (puede tardar un momento)...")

for ticker in retornos_activos.columns:
    # Beta histórica (apalancada) = Cov(activo, mercado) / Var(mercado)
    covarianza_activo_mercado = retornos_activos[ticker].cov(retornos_diarios['Market']) * 252
    beta_apalancada = covarianza_activo_mercado / varianza_mercado

    # Retorno esperado usando CAPM tradicional
    retorno_esperado_capm = tasa_libre_riesgo_anual + beta_apalancada * prima_mercado

    # Ratio real de Deuda/Capital (D/E). Si no hay dato válido se deja NaN (sin valores inventados).
    # Nota: para bancos/aseguradoras el D/E no es económicamente significativo.
    relacion_deuda_capital = np.nan
    try:
        de_ratio = yf.Ticker(ticker.upper()).info.get('debtToEquity')
        # yfinance entrega el ratio en porcentaje; D/E negativo (capital contable negativo) no es válido para Hamada
        if de_ratio is not None and de_ratio >= 0:
            relacion_deuda_capital = de_ratio / 100.0
    except Exception:
        pass

    # Beta desapalancada de Hamada (riesgo del negocio sin deuda): B_U = B_L / (1 + (1 - t) * D/E)
    beta_desapalancada = beta_apalancada / (1 + (1 - tasa_impuestos) * relacion_deuda_capital)

    metricas_acciones.append({
        'Activo': ticker,
        'Beta_Apalancada': beta_apalancada,
        'Retorno_Esperado_CAPM': retorno_esperado_capm,
        'Relacion_D_E': relacion_deuda_capital,
        'DE_Disponible': not np.isnan(relacion_deuda_capital),
        'Beta_Hamada_Desapalancada': beta_desapalancada,
    })

df_metricas_fundamentales = pd.DataFrame(metricas_acciones)

# La beta desapalancada no se usa directo en el CAPM: se re-apalanca a una estructura de
# capital objetivo (aquí, la mediana del D/E del universo) para obtener la beta del accionista.
de_objetivo = df_metricas_fundamentales['Relacion_D_E'].median()
df_metricas_fundamentales['Beta_Reapalancada'] = (
    df_metricas_fundamentales['Beta_Hamada_Desapalancada'] * (1 + (1 - tasa_impuestos) * de_objetivo)
)
df_metricas_fundamentales['Retorno_Esperado_Hamada'] = (
    tasa_libre_riesgo_anual + df_metricas_fundamentales['Beta_Reapalancada'] * prima_mercado
)

print(f"Retorno de mercado anualizado (SPY): {retorno_mercado_anual:.2%} | D/E objetivo (mediana): {de_objetivo:.2f}")
print(f"Acciones sin D/E válido: {(~df_metricas_fundamentales['DE_Disponible']).sum()}")
print("--- Análisis de CAPM y Beta de Hamada ---")
display(df_metricas_fundamentales)

# Paso 3: Análisis Técnico con Indicadores

### Fase 3.1: Medias Móviles, RSI, MACD y Oscilador Estocástico

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


def calcular_rsi(precios, n=14):
    """RSI de Wilder (suavizado exponencial alpha = 1/n). Se usa en todo el notebook."""
    delta = precios.diff()
    ganancia = delta.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    perdida = (-delta.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
    rs = ganancia / perdida
    return 100 - (100 / (1 + rs))


# Escáner: Buscar acciones saliendo de sobreventa
acciones_sobreventa = []
indicadores_sobreventa = {}

for ticker, df_accion in acciones_validas.items():
    df_tecnico = df_accion.copy()
    df_tecnico['Date'] = pd.to_datetime(df_tecnico['Date'])
    df_tecnico = df_tecnico.sort_values('Date').set_index('Date')

    # RSI 14 (Wilder)
    df_tecnico['RSI_14'] = calcular_rsi(df_tecnico['Close'], 14)

    # Oscilador Estocástico
    minimo_14 = df_tecnico['Low'].rolling(window=14).min()
    maximo_14 = df_tecnico['High'].rolling(window=14).max()
    # Evitar división entre cero si el máximo y el mínimo de 14 días coinciden
    rango_14 = (maximo_14 - minimo_14).replace(0, np.nan)
    df_tecnico['Estocastico_K'] = 100 * ((df_tecnico['Close'] - minimo_14) / rango_14)
    df_tecnico['Estocastico_D'] = df_tecnico['Estocastico_K'].rolling(window=3).mean()

    # MACD
    ema_12 = df_tecnico['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df_tecnico['Close'].ewm(span=26, adjust=False).mean()
    df_tecnico['MACD'] = ema_12 - ema_26
    df_tecnico['MACD_Senal'] = df_tecnico['MACD'].ewm(span=9, adjust=False).mean()

    # Medias Móviles Simples (SMA 20 y 50)
    df_tecnico['SMA_20'] = df_tecnico['Close'].rolling(window=20).mean()
    df_tecnico['SMA_50'] = df_tecnico['Close'].rolling(window=50).mean()

    indicadores_sobreventa[ticker] = df_tecnico

    # Condiciones de salida de sobreventa (últimos 2 días)
    if len(df_tecnico) > 2:
        ultimo_rsi = df_tecnico['RSI_14'].iloc[-1]
        penultimo_rsi = df_tecnico['RSI_14'].iloc[-2]

        ultima_K = df_tecnico['Estocastico_K'].iloc[-1]
        ultima_D = df_tecnico['Estocastico_D'].iloc[-1]
        penultima_K = df_tecnico['Estocastico_K'].iloc[-2]
        penultima_D = df_tecnico['Estocastico_D'].iloc[-2]

        # 1. RSI estaba cerca o debajo de 30 y ahora está subiendo (pero aún por debajo de 45)
        criterio_rsi = (ultimo_rsi > penultimo_rsi) and (ultimo_rsi < 45) and (penultimo_rsi < 35)

        # 2. Estocástico cruzando al alza en zona de sobreventa (< 30)
        criterio_estocastico = (ultima_K > ultima_D) and (penultima_K <= penultima_D) and (ultima_K < 30)

        if criterio_rsi or criterio_estocastico:
            acciones_sobreventa.append(ticker)

print("Acciones que muestran perfil 'Saliendo de Sobreventa':")
print(acciones_sobreventa)

if acciones_sobreventa:
    activo_tecnico = acciones_sobreventa[0]
    print(f"\nMostrando el gráfico de la primera acción encontrada: {activo_tecnico.upper()}")
else:
    activo_tecnico = list(acciones_validas.keys())[0]
    print(f"\nNo se encontraron acciones con ese perfil exacto hoy. Mostrando: {activo_tecnico.upper()}")

df_tecnico = indicadores_sobreventa[activo_tecnico]

# Visualización gráfica de los indicadores
df_grafico = df_tecnico.tail(252) # Último año bursátil
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Gráfico 1: Precios y Medias Móviles
ax1.plot(df_grafico.index, df_grafico['Close'], label='Precio Cierre', color='black')
ax1.plot(df_grafico.index, df_grafico['SMA_20'], label='SMA 20', color='blue')
ax1.plot(df_grafico.index, df_grafico['SMA_50'], label='SMA 50', color='red')
ax1.set_title(f'Indicadores Técnicos para {activo_tecnico.upper()} - Perfil de Sobreventa')
ax1.legend()
ax1.grid(True)

# Gráfico 2: RSI
ax2.plot(df_grafico.index, df_grafico['RSI_14'], color='purple', label='RSI 14')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
ax2.set_ylabel('RSI')
ax2.legend()
ax2.grid(True)

# Gráfico 3: MACD
ax3.plot(df_grafico.index, df_grafico['MACD'], color='blue', label='MACD')
ax3.plot(df_grafico.index, df_grafico['MACD_Senal'], color='orange', label='Señal')
ax3.axhline(0, color='gray', linestyle='--')
ax3.legend()
ax3.grid(True)

# Gráfico 4: Estocástico
ax4.plot(df_grafico.index, df_grafico['Estocastico_K'], color='cyan', label='%K')
ax4.plot(df_grafico.index, df_grafico['Estocastico_D'], color='magenta', label='%D')
ax4.axhline(80, color='red', linestyle='--', alpha=0.5)
ax4.axhline(20, color='green', linestyle='--', alpha=0.5)
ax4.legend()
ax4.grid(True)

plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

### Fase 3.2: Categorización de Activos por Volatilidad y Direccionalidad
En esta fase clasificamos dinámicamente las acciones del universo analizado de acuerdo con dos dimensiones clave:
1. **Volatilidad Anualizada**: Clasificadas en *Alta Volatilidad* o *Baja Volatilidad* utilizando el valor medio de volatilidad como umbral.
2. **Direccionalidad (RSI)**: Clasificadas en tendencia *Alcista* (RSI de 14 días > 50) o *Bajista* (RSI de 14 días <= 50).

In [ ]:
# 1. Calcular volatilidad histórica anualizada para todos los activos
volatilidades = retornos_activos.std() * np.sqrt(252)
umbral_volatilidad = volatilidades.median()

# 2. Calcular el último RSI_14 para cada activo (retornos_activos se define en la Fase 1.2)
rsis_actuales = {}
for ticker, df_accion in acciones_validas.items():
    if ticker not in retornos_activos.columns:
        continue
    precios_close = df_accion.sort_values('Date')['Close']
    # RSI de Wilder (misma función que en la Fase 3.1)
    rsis_actuales[ticker] = calcular_rsi(precios_close, 14).iloc[-1]

# 3. Clasificar según condiciones cuantitativas
categorias_acciones = []
for ticker in volatilidades.index:
    vol = volatilidades[ticker]
    rsi = rsis_actuales.get(ticker, 50.0)

    # Determinar categoría de volatilidad
    cat_vol = "Alta Volatilidad" if vol > umbral_volatilidad else "Baja Volatilidad"

    # Determinar dirección de la tendencia
    cat_dir = "Alcista" if rsi > 50 else "Bajista"

    categorias_acciones.append({
        'Activo': ticker,
        'Volatilidad_Anualizada': vol,
        'RSI_Actual': rsi,
        'Categoria_Volatilidad': cat_vol,
        'Direccionalidad': cat_dir,
        'Cuadrante': f"{cat_vol} - {cat_dir}"
    })

df_categorizacion = pd.DataFrame(categorias_acciones)
print("--- Clasificación Cuantitativa de Activos ---")
display(df_categorizacion.sort_values(by='Cuadrante'))

### Fase 3.3: Filtro de Régimen de Mercado — SPY vs SMA 200 (Mejora 2)
Regla del backtest: **a fin de cada mes**, si SPY cierra por encima de su media móvil de 200 días se invierte en el portafolio del Top 5; si cierra por debajo, se pasa a **efectivo (T-Bills)** hasta que vuelva arriba.

En el backtest de 20 años este filtro evitó la caída de 2008 (−0.5 % en lugar de −61 %) y redujo la caída máxima de −68 % a −35 %. No protege en caídas rápidas (2022).

In [ ]:
# MEJORA 2: filtro de régimen de mercado con SPY vs su media móvil de 200 días
spy_close = datos_spy.set_index(pd.to_datetime(datos_spy['Date']))['Close']
spy_sma200 = spy_close.rolling(200).mean()
ultimo_spy, ultima_sma200 = spy_close.iloc[-1], spy_sma200.iloc[-1]
mercado_alcista = bool(ultimo_spy > ultima_sma200)
distancia_sma200 = ultimo_spy / ultima_sma200 - 1

# Último cierre de mes (la regla se revisa a fin de mes)
fin_mes = spy_close.resample('ME').last().index[-2]
fin_mes = spy_close.loc[:fin_mes].index[-1]
alcista_fin_mes = bool(spy_close.loc[fin_mes] > spy_sma200.loc[fin_mes])

plt.figure(figsize=(12, 4))
plt.plot(spy_close.index[-756:], spy_close.tail(756), color='black', label='SPY')
plt.plot(spy_sma200.index[-756:], spy_sma200.tail(756), color='orange', label='SMA 200')
plt.fill_between(spy_close.index[-756:], spy_close.tail(756).min(), spy_close.tail(756).max(),
                 where=(spy_close.tail(756) < spy_sma200.tail(756)), color='red', alpha=0.08, label='SPY < SMA 200')
plt.title('Filtro de mercado: SPY vs SMA 200 (últimos 3 años)')
plt.legend(); plt.grid(True); plt.show()

print(f"SPY: {ultimo_spy:,.2f} | SMA 200: {ultima_sma200:,.2f} | distancia: {distancia_sma200:+.2%}")
print(f"Régimen hoy: {'ALCISTA → invertir en el Top 5' if mercado_alcista else 'BAJISTA → efectivo / T-Bills'}")
print(f"Régimen al último cierre de mes ({fin_mes.date()}): {'alcista' if alcista_fin_mes else 'bajista'} "
      f"(la regla del backtest decide con este valor)")

# Paso 4: Análisis Econométrico y Riesgo/Rendimiento

### Fase 4.1: Retornos, Volatilidad, Matriz de Correlación y Cointegración

In [ ]:
import seaborn as sns
from statsmodels.tsa.stattools import coint

# retornos_activos (acciones sin 'Market') ya está definido en la Fase 1.2

# Métricas de riesgo y rendimiento
retorno_medio_anual = retornos_activos.mean() * 252
varianza_anual = retornos_activos.var() * 252
volatilidad_anual = retornos_activos.std() * np.sqrt(252)

# Ratio de Sharpe
ratio_sharpe = (retorno_medio_anual - tasa_libre_riesgo_anual) / volatilidad_anual

# Resumen de métricas econométricas
df_econometrico = pd.DataFrame({
    'Retorno_Medio_Anual': retorno_medio_anual,
    'Varianza_Anual': varianza_anual,
    'Volatilidad_Anual': volatilidad_anual,
    'Ratio_Sharpe': ratio_sharpe
})

print("--- Métricas Econométricas de Activos Individuales ---")
display(df_econometrico)

# Matriz de covarianza y correlación
matriz_cov = retornos_activos.cov() * 252
matriz_corr = retornos_activos.corr()

# Visualizar mapa de calor de correlaciones (sin anotaciones: con cientos de activos serían ilegibles)
plt.figure(figsize=(10, 8))
sns.heatmap(matriz_corr, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=False, yticklabels=False)
plt.title('Matriz de Correlación de Retornos')
plt.show()

# Test de Cointegración (ilustrativo): solo el primer par de columnas, no es una búsqueda de pares
simbolos = list(retornos_activos.columns)
if len(simbolos) >= 2:
    activo1, activo2 = simbolos[0], simbolos[1]
    puntos_precio1 = df_precios_unidos[activo1]
    puntos_precio2 = df_precios_unidos[activo2]
    p_valor = coint(puntos_precio1, puntos_precio2)[1]

    print(f"\n--- Test de Cointegración: {activo1} vs {activo2} ---")
    print(f"p-valor: {p_valor:.4f}")
    if p_valor < 0.05:
        print("Existe evidencia de que los activos están cointegrados.")
    else:
        print("No hay evidencia sólida de cointegración.")

# Paso 5: Construcción y Análisis de Portafolios

### Fase 5.1: Portafolio Equiponderado

In [ ]:
# Crear pesos iguales para todos los activos
num_activos = len(retornos_activos.columns)
pesos_iguales = np.array([1 / num_activos] * num_activos)

# Calcular rendimiento del portafolio equiponderado
rendimiento_portafolio = np.dot(pesos_iguales, retorno_medio_anual)

# Calcular varianza y volatilidad del portafolio
varianza_portafolio = np.dot(pesos_iguales.T, np.dot(matriz_cov, pesos_iguales))
volatilidad_portafolio = np.sqrt(varianza_portafolio)

# Sharpe Ratio
ratio_sharpe_portafolio = (rendimiento_portafolio - tasa_libre_riesgo_anual) / volatilidad_portafolio

print("--- Métricas del Portafolio Equiponderado ---")
print(f"Retorno Esperado Anualizado: {rendimiento_portafolio:.4f}")
print(f"Volatilidad Anualizada: {volatilidad_portafolio:.4f}")
print(f"Ratio de Sharpe: {ratio_sharpe_portafolio:.4f}")

# Paso 6: Optimización de Portafolio con SciPy

### Fase 6.1: Frontera Eficiente, GMV y Máximo Sharpe

In [ ]:
import scipy.optimize as sco

# Función para calcular estadísticas del portafolio dado unos pesos
def calcular_estadisticas_portafolio(pesos):
    pesos = np.array(pesos)
    ret = np.dot(pesos, retorno_medio_anual)
    vol = np.sqrt(np.dot(pesos.T, np.dot(matriz_cov, pesos)))
    sharpe = (ret - tasa_libre_riesgo_anual) / vol
    return np.array([ret, vol, sharpe])

# Funciones a minimizar para las optimizaciones
def minimizar_sharpe(pesos):
    return -calcular_estadisticas_portafolio(pesos)[2]

def minimizar_volatilidad(pesos):
    return calcular_estadisticas_portafolio(pesos)[1]

# Restricciones de pesos (suma de pesos = 1, y límites individuales del 35%)
restricciones = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
limites = tuple((0, 0.35) for _ in range(num_activos))
pesos_iniciales = num_activos * [1. / num_activos,]

# 1. Optimización Máximo Sharpe
opt_max_sharpe = sco.minimize(minimizar_sharpe, pesos_iniciales, method='SLSQP', bounds=limites, constraints=restricciones)
if not opt_max_sharpe.success:
    print(f"AVISO: la optimización de Máximo Sharpe no convergió: {opt_max_sharpe.message}")
pesos_max_sharpe = opt_max_sharpe.x
stats_max_sharpe = calcular_estadisticas_portafolio(pesos_max_sharpe)

# 2. Optimización Mínima Varianza Global (GMV)
opt_gmv = sco.minimize(minimizar_volatilidad, pesos_iniciales, method='SLSQP', bounds=limites, constraints=restricciones)
if not opt_gmv.success:
    print(f"AVISO: la optimización GMV no convergió: {opt_gmv.message}")
pesos_gmv = opt_gmv.x
stats_gmv = calcular_estadisticas_portafolio(pesos_gmv)


# 3. Construcción de la Frontera Eficiente
def retorno_maximo_alcanzable(retornos, limite):
    """Mayor retorno posible con pesos <= limite: llenar primero los activos de mayor retorno."""
    restante, total = 1.0, 0.0
    for r in np.sort(np.asarray(retornos))[::-1]:
        w = min(limite, restante)
        total += w * r
        restante -= w
        if restante <= 1e-12:
            break
    return total


limite_peso = 0.35
r_max = retorno_maximo_alcanzable(retorno_medio_anual, limite_peso)
rendimientos_objetivo = np.linspace(stats_gmv[0], r_max, 50)
rendimientos_frontera = []
volatilidades_frontera = []

for r_obj in rendimientos_objetivo:
    rest_ef = (
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
        {'type': 'eq', 'fun': lambda x, r=r_obj: calcular_estadisticas_portafolio(x)[0] - r}
    )
    opt_ef = sco.minimize(minimizar_volatilidad, pesos_iniciales, method='SLSQP', bounds=limites, constraints=rest_ef)
    # Solo se grafican los puntos donde el optimizador realmente encontró solución
    if opt_ef.success:
        rendimientos_frontera.append(r_obj)
        volatilidades_frontera.append(opt_ef.fun)

print(f"Frontera eficiente: {len(rendimientos_frontera)} de {len(rendimientos_objetivo)} puntos resueltos.")

# Mostrar Resultados Comparativos
df_comparativo = pd.DataFrame({
    'Portafolio': ['Equiponderado', 'GMV', 'Max Sharpe'],
    'Retorno_Esperado': [rendimiento_portafolio, stats_gmv[0], stats_max_sharpe[0]],
    'Volatilidad': [volatilidad_portafolio, stats_gmv[1], stats_max_sharpe[1]],
    'Ratio_Sharpe': [ratio_sharpe_portafolio, stats_gmv[2], stats_max_sharpe[2]]
})
display(df_comparativo)

# Graficar la Frontera Eficiente
plt.figure(figsize=(12, 8))
plt.plot(volatilidades_frontera, rendimientos_frontera, 'b--', label='Frontera Eficiente')
plt.scatter(volatilidad_anual, retorno_medio_anual, color='gray', label='Acciones Individuales')
plt.scatter(volatilidad_portafolio, rendimiento_portafolio, marker='s', color='orange', s=100, label='Equiponderado')
plt.scatter(stats_gmv[1], stats_gmv[0], marker='v', color='green', s=100, label='GMV')
plt.scatter(stats_max_sharpe[1], stats_max_sharpe[0], marker='*', color='red', s=150, label='Max Sharpe')
plt.title('Optimización de Portafolio y Frontera Eficiente')
plt.xlabel('Volatilidad Anualizada')
plt.ylabel('Retorno Anualizado')
plt.legend()
plt.grid(True)
plt.show()

# Paso 7: Simulaciones de Monte Carlo y Recomendaciones Finales

### Fase 7.1: Simulación de Portafolios por Monte Carlo

In [ ]:
num_simulaciones = 5000
resultados_simulados = np.zeros((3, num_simulaciones))
rng = np.random.default_rng(42)  # semilla para reproducibilidad


def recortar_pesos(pesos, limite, max_iter=100):
    """Recorta los pesos que excedan el límite y reparte el excedente entre el resto."""
    pesos = pesos.copy()
    for _ in range(max_iter):
        exceso = pesos > limite
        if not exceso.any():
            break
        sobrante = (pesos[exceso] - limite).sum()
        pesos[exceso] = limite
        libres = pesos < limite
        pesos[libres] += sobrante * pesos[libres] / pesos[libres].sum()
    return pesos


# Pesos aleatorios con distribución de Dirichlet (alpha pequeño => portafolios concentrados).
# Con cientos de activos, pesos uniformes normalizados colapsarían al portafolio equiponderado.
for i in range(num_simulaciones):
    pesos_aleatorios = rng.dirichlet(np.full(num_activos, 0.05))
    pesos_aleatorios = recortar_pesos(pesos_aleatorios, 0.35)  # suman 1 y ninguno supera el 35%

    resultados_simulados[:, i] = calcular_estadisticas_portafolio(pesos_aleatorios)

# Graficar nube de puntos simulados
plt.figure(figsize=(10, 6))
plt.scatter(resultados_simulados[1], resultados_simulados[0], c=resultados_simulados[2], cmap='viridis', marker='o', s=10, alpha=0.3)
plt.colorbar(label='Ratio de Sharpe')
plt.title('Simulación de Monte Carlo de Portafolios')
plt.xlabel('Volatilidad Anualizada')
plt.ylabel('Retorno Esperado Anualizado')
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# 1. Preparar las métricas de cada acción
metricas = []

for ticker in retornos_activos.columns:
    # Rendimiento y Sharpe ya calculados en el entorno
    rendimiento = retorno_medio_anual[ticker]
    sharpe = ratio_sharpe[ticker]
    vol = volatilidad_anual[ticker]

    # Calcular Control Drawdown (calculando Max Drawdown primero)
    precios = df_precios_unidos[ticker]
    maximos_historicos = precios.cummax()
    drawdowns = (precios - maximos_historicos) / maximos_historicos
    max_drawdown = drawdowns.min()  # suele ser un valor negativo
    # Convertimos a Control Drawdown: cuanto menor sea la caída máxima, mayor será este valor
    control_drawdown = 1 + max_drawdown

    # Baja Volatilidad (inversa de la volatilidad, para que a menor vol, mayor puntaje)
    baja_volatilidad = 1 / vol if vol > 0 else 0

    metricas.append({
        'Ticker': ticker,
        'Rendimiento': rendimiento,
        'Sharpe': sharpe,
        'Control_Drawdown': control_drawdown,
        'Baja_Volatilidad': baja_volatilidad
    })

df_metricas = pd.DataFrame(metricas)

# 2. Normalizar métricas (Min-Max) para tenerlas en una escala de 0 a 1
def min_max_norm(serie):
    if serie.max() == serie.min():
        return 1.0
    return (serie - serie.min()) / (serie.max() - serie.min())

df_metricas['Norm_Sharpe'] = min_max_norm(df_metricas['Sharpe'])
df_metricas['Norm_Rendimiento'] = min_max_norm(df_metricas['Rendimiento'])
df_metricas['Norm_Drawdown'] = min_max_norm(df_metricas['Control_Drawdown'])
df_metricas['Norm_Baja_Vol'] = min_max_norm(df_metricas['Baja_Volatilidad'])

# 3. Calcular Score General
# 35% Sharpe + 25% Rendimiento + 25% Control Drawdown + 15% Baja Volatilidad
df_metricas['Score_General'] = (
    0.35 * df_metricas['Norm_Sharpe'] +
    0.25 * df_metricas['Norm_Rendimiento'] +
    0.25 * df_metricas['Norm_Drawdown'] +
    0.15 * df_metricas['Norm_Baja_Vol']
)

# 4. Obtener las top 5 y generar el reporte
top_5 = df_metricas.sort_values(by='Score_General', ascending=False).head(5)

print("=" * 70)
print("                INFORME AUTOMATIZADO: TOP 5 ACCIONES")
print("=" * 70)
print("Fórmula:\nScore General = 35% Sharpe + 25% Rendimiento + 25% Control Drawdown + 15% Baja Volatilidad\n")

reporte = top_5[['Ticker', 'Rendimiento', 'Sharpe', 'Control_Drawdown', 'Baja_Volatilidad', 'Score_General']].copy()
reporte.rename(columns={
    'Rendimiento': 'Retorno Anual',
    'Control_Drawdown': 'Ctrl Drawdown',
    'Baja_Volatilidad': 'Factor Baja Vol'
}, inplace=True)

display(reporte.style.format({
    'Retorno Anual': '{:.2%}',
    'Sharpe': '{:.2f}',
    'Ctrl Drawdown': '{:.2%}',
    'Factor Baja Vol': '{:.2f}',
    'Score_General': '{:.4f}'
}).hide(axis='index'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.optimize as sco
from statsmodels.tsa.stattools import coint

# 0. Extraer los top 5 tickers
top_tickers = top_5['Ticker'].tolist()
print("=" * 70)
print(f"EJECUTANDO ANÁLISIS INTEGRAL PARA EL TOP 5: {top_tickers}")
print("=" * 70)

# 1. Filtrar retornos y calcular métricas base
retornos_top5 = retornos_activos[top_tickers]
retorno_medio_top5 = retornos_top5.mean() * 252
matriz_cov_top5 = retornos_top5.cov() * 252

# 2. CAPM y Beta
print("\n--- 1. Análisis CAPM y Beta (Top 5) ---")
metricas_capm_top5 = []
for ticker in top_tickers:
    cov_mercado = retornos_activos[ticker].cov(retornos_diarios['Market']) * 252
    beta = cov_mercado / (retornos_diarios['Market'].var() * 252)
    ret_esperado = tasa_libre_riesgo_anual + beta * (retorno_mercado_anual - tasa_libre_riesgo_anual)
    metricas_capm_top5.append({'Activo': ticker, 'Beta': beta, 'Retorno_Esperado_CAPM': ret_esperado})
display(pd.DataFrame(metricas_capm_top5))

# 3. Análisis Técnico del Líder (la acción #1)
lider = top_tickers[0]
print(f"\n--- 2. Análisis Técnico Rápido del Líder: {lider.upper()} ---")
df_lider = df_precios_unidos[[lider]].rename(columns={lider: 'Close'}).copy()
df_lider['SMA_20'] = df_lider['Close'].rolling(20).mean()
df_lider['SMA_50'] = df_lider['Close'].rolling(50).mean()

plt.figure(figsize=(10, 4))
plt.plot(df_lider.index[-252:], df_lider['Close'].tail(252), label='Precio Cierre', color='black')
plt.plot(df_lider.index[-252:], df_lider['SMA_20'].tail(252), label='SMA 20', color='blue')
plt.plot(df_lider.index[-252:], df_lider['SMA_50'].tail(252), label='SMA 50', color='red')
plt.title(f'Tendencia Reciente (Último Año) - {lider.upper()}')
plt.xlabel('Fecha')
plt.ylabel('Precio')
plt.legend()
plt.grid(True)
plt.show()

# 4. Matriz de Correlación Top 5
print("\n--- 3. Matriz de Correlación Econométrica (Top 5) ---")
plt.figure(figsize=(6, 5))
sns.heatmap(retornos_top5.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlación entre el Top 5')
plt.show()

# 5. Optimización de Portafolio (Max Sharpe)
print("\n--- 4. Optimización de Portafolio (Top 5) ---")
num_activos_top5 = len(top_tickers)

def calc_stats_top5(pesos):
    pesos = np.array(pesos)
    ret = np.dot(pesos, retorno_medio_top5)
    vol = np.sqrt(np.dot(pesos.T, np.dot(matriz_cov_top5, pesos)))
    sharpe = (ret - tasa_libre_riesgo_anual) / vol
    return np.array([ret, vol, sharpe])

def min_sharpe_top5(pesos):
    return -calc_stats_top5(pesos)[2]

# Restricciones y límites (0 a 100% por ser portafolio concentrado)
restricciones_top5 = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
limites_top5 = tuple((0, 1.0) for _ in range(num_activos_top5))
pesos_ini_top5 = num_activos_top5 * [1. / num_activos_top5,]

opt_sharpe_top5 = sco.minimize(min_sharpe_top5, pesos_ini_top5, method='SLSQP', bounds=limites_top5, constraints=restricciones_top5)
stats_opt = calc_stats_top5(opt_sharpe_top5.x)

print("Distribución Óptima (Máximo Sharpe Ratio):")
for t, p in zip(top_tickers, opt_sharpe_top5.x):
    print(f"  - {t.upper()}: {p:.2%}")
print(f"\nRetorno Esperado del Portafolio: {stats_opt[0]:.2%}")
print(f"Volatilidad del Portafolio: {stats_opt[1]:.2%}")
print(f"Sharpe Ratio: {stats_opt[2]:.4f}")

# 6. Simulación Monte Carlo
print("\n--- 5. Simulación de Monte Carlo (Top 5) ---")
num_sims = 3000
rng_top5 = np.random.default_rng(42)  # semilla para reproducibilidad
res_sims = np.zeros((3, num_sims))
for i in range(num_sims):
    w = rng_top5.random(num_activos_top5)
    w /= np.sum(w)
    s = calc_stats_top5(w)
    res_sims[:, i] = s

plt.figure(figsize=(9, 6))
plt.scatter(res_sims[1], res_sims[0], c=res_sims[2], cmap='viridis', marker='o', s=10, alpha=0.5)
plt.colorbar(label='Ratio de Sharpe')
plt.scatter(stats_opt[1], stats_opt[0], color='red', marker='*', s=250, label='Portafolio Óptimo (Max Sharpe)')
plt.title('Simulación Monte Carlo - Portafolios Top 5')
plt.xlabel('Volatilidad Anualizada')
plt.ylabel('Retorno Esperado Anualizado')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("=" * 70)
print("     RESUMEN INTEGRAL Y ANÁLISIS TÉCNICO: TOP 5 ACCIONES")
print("=" * 70)

# ------------------------------------------------------------------
# 1. Indicadores técnicos de cada acción del Top 5
# ------------------------------------------------------------------
tecnicos_top5 = {}
for ticker in top_tickers:
    # Precios desde df_precios_unidos: contiene siempre las acciones del ranking
    # (acciones_validas puede cambiar si la celda de descarga se re-ejecuta y Yahoo omite tickers)
    df_tec = df_precios_unidos[[ticker]].rename(columns={ticker: 'Close'}).copy()

    # Medias Móviles Simples (SMA 20 y 50)
    df_tec['SMA_20'] = df_tec['Close'].rolling(window=20).mean()
    df_tec['SMA_50'] = df_tec['Close'].rolling(window=50).mean()

    # RSI 14 (Wilder, misma función que en la Fase 3.1)
    df_tec['RSI_14'] = calcular_rsi(df_tec['Close'], 14)

    # MACD
    ema_12 = df_tec['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df_tec['Close'].ewm(span=26, adjust=False).mean()
    df_tec['MACD'] = ema_12 - ema_26
    df_tec['MACD_Senal'] = df_tec['MACD'].ewm(span=9, adjust=False).mean()

    tecnicos_top5[ticker] = df_tec

# ------------------------------------------------------------------
# 2. Tabla con todas las métricas por acción (filas = métricas, columnas = acciones)
# ------------------------------------------------------------------
fund = df_metricas_fundamentales.set_index('Activo')
categ = df_categorizacion.set_index('Activo')
met = top_5.set_index('Ticker')
pesos_top5_ms = pd.Series(opt_sharpe_top5.x, index=top_tickers)
pesos_univ_ms = pd.Series(pesos_max_sharpe, index=retornos_activos.columns)
pesos_univ_gmv = pd.Series(pesos_gmv, index=retornos_activos.columns)


def pct(x):
    return '—' if pd.isna(x) else f'{x:.2%}'


def num(x, d=2):
    return '—' if pd.isna(x) else f'{x:.{d}f}'


def arriba_abajo(a, b):
    return 'Arriba' if a > b else 'Abajo'


columnas = {}
for rank, t in enumerate(top_tickers, start=1):
    u = tecnicos_top5[t].iloc[-1]
    columnas[f'#{rank} {t.upper()}'] = {
        'Score General': num(met.loc[t, 'Score_General'], 4),
        '▸ RIESGO / RENDIMIENTO': '',
        'Retorno anual': pct(met.loc[t, 'Rendimiento']),
        'Volatilidad anual': pct(volatilidad_anual[t]),
        'Ratio de Sharpe': num(met.loc[t, 'Sharpe']),
        'Max Drawdown': pct(met.loc[t, 'Control_Drawdown'] - 1),
        '▸ FUNDAMENTAL (CAPM / HAMADA)': '',
        'Beta (apalancada)': num(fund.loc[t, 'Beta_Apalancada']),
        'Retorno esperado CAPM': pct(fund.loc[t, 'Retorno_Esperado_CAPM']),
        'Deuda / Capital (D/E)': num(fund.loc[t, 'Relacion_D_E']),
        'Beta desapalancada (Hamada)': num(fund.loc[t, 'Beta_Hamada_Desapalancada']),
        'Retorno esperado Hamada': pct(fund.loc[t, 'Retorno_Esperado_Hamada']),
        '▸ TÉCNICO (último cierre)': '',
        'Precio de cierre': num(u['Close']),
        'RSI 14': num(u['RSI_14'], 1),
        'MACD vs Señal': 'Alcista' if u['MACD'] > u['MACD_Senal'] else 'Bajista',
        'Precio vs SMA 20': arriba_abajo(u['Close'], u['SMA_20']),
        'Precio vs SMA 50': arriba_abajo(u['Close'], u['SMA_50']),
        'Cuadrante': categ.loc[t, 'Cuadrante'],
        '▸ PESOS DE PORTAFOLIO': '',
        'Peso Max Sharpe (Top 5)': pct(pesos_top5_ms[t]),
        'Peso Equiponderado (Top 5)': pct(1 / len(top_tickers)),
        'Peso Max Sharpe (universo)': pct(pesos_univ_ms[t]),
        'Peso GMV (universo)': pct(pesos_univ_gmv[t]),
    }

tabla_top5 = pd.DataFrame(columnas)
print("\n--- Métricas completas del Top 5 ---")
display(tabla_top5.style.set_properties(**{'text-align': 'right'}))

# ------------------------------------------------------------------
# 3. Resumen de portafolios (retorno, volatilidad, Sharpe, beta y max drawdown)
# ------------------------------------------------------------------
def calcular_max_drawdown_portafolio(pesos, df_retornos):
    """Calcula el Max Drawdown histórico de un portafolio basado en sus pesos y retornos diarios."""
    # Retornos diarios del portafolio
    retornos_port = df_retornos.dot(pesos)
    # Curva de valor acumulado del portafolio (empezando en 1)
    riqueza_acumulada = (1 + retornos_port).cumprod()
    maximos_historicos = riqueza_acumulada.cummax()
    drawdowns = (riqueza_acumulada - maximos_historicos) / maximos_historicos
    return drawdowns.min()

betas_univ = fund['Beta_Apalancada'].reindex(retornos_activos.columns).values
betas_top5 = fund.loc[top_tickers, 'Beta_Apalancada'].values
pesos_eq_top5 = np.full(len(top_tickers), 1 / len(top_tickers))
pesos_eq_univ = np.full(len(betas_univ), 1 / len(betas_univ))

# Calcular Max Drawdown para cada portafolio
mdd_top5_ms = calcular_max_drawdown_portafolio(pesos_top5_ms.values, retornos_top5)
mdd_top5_eq = calcular_max_drawdown_portafolio(pesos_eq_top5, retornos_top5)
mdd_univ_ms = calcular_max_drawdown_portafolio(pesos_max_sharpe, retornos_activos)
mdd_univ_gmv = calcular_max_drawdown_portafolio(pesos_gmv, retornos_activos)
mdd_univ_eq = calcular_max_drawdown_portafolio(pesos_eq_univ, retornos_activos)

df_portafolios = pd.DataFrame(
    [
        [*stats_opt, pesos_top5_ms.values @ betas_top5, mdd_top5_ms],
        [*calc_stats_top5(pesos_eq_top5), pesos_eq_top5 @ betas_top5, mdd_top5_eq],
        [*stats_max_sharpe, pesos_max_sharpe @ betas_univ, mdd_univ_ms],
        [*stats_gmv, pesos_gmv @ betas_univ, mdd_univ_gmv],
        [rendimiento_portafolio, volatilidad_portafolio, ratio_sharpe_portafolio, pesos_eq_univ @ betas_univ, mdd_univ_eq],
    ],
    index=['Top 5 - Max Sharpe', 'Top 5 - Equiponderado', 'Universo - Max Sharpe',
           'Universo - GMV', 'Universo - Equiponderado'],
    columns=['Retorno esperado', 'Volatilidad', 'Ratio de Sharpe', 'Beta', 'Max Drawdown'],
)
print("\n--- Resumen de Portafolios ---")
display(df_portafolios.style.format({
    'Retorno esperado': '{:.2%}',
    'Volatilidad': '{:.2%}',
    'Ratio de Sharpe': '{:.2f}',
    'Beta': '{:.2f}',
    'Max Drawdown': '{:.2%}',
}))

# ------------------------------------------------------------------
# 4. Gráficos técnicos de cada acción del Top 5
# ------------------------------------------------------------------
for rank, ticker in enumerate(top_tickers, start=1):
    # Filtrar solo el último año bursátil para una mejor visualización (252 días)
    df_graf = tecnicos_top5[ticker].tail(252)

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # Gráfico 1: Precio y Medias Móviles
    ax1.plot(df_graf.index, df_graf['Close'], label='Precio Cierre', color='black')
    ax1.plot(df_graf.index, df_graf['SMA_20'], label='SMA 20', color='blue')
    ax1.plot(df_graf.index, df_graf['SMA_50'], label='SMA 50', color='red')
    ax1.set_title(f'#{rank} {ticker.upper()} - Score {met.loc[ticker, "Score_General"]:.3f} | '
                  f'Peso Max Sharpe Top 5: {pesos_top5_ms[ticker]:.1%}')
    ax1.set_ylabel('Precio')
    ax1.legend()
    ax1.grid(True)

    # Gráfico 2: RSI
    ax2.plot(df_graf.index, df_graf['RSI_14'], color='purple', label='RSI (14)')
    ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
    ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
    ax2.set_ylabel('RSI')
    ax2.legend(loc='upper left')
    ax2.grid(True)

    # Gráfico 3: MACD
    ax3.plot(df_graf.index, df_graf['MACD'], color='blue', label='MACD')
    ax3.plot(df_graf.index, df_graf['MACD_Senal'], color='orange', label='Señal')
    ax3.axhline(0, color='gray', linestyle='--')
    ax3.set_ylabel('MACD')
    ax3.set_xlabel('Fecha')
    ax3.legend(loc='upper left')
    ax3.grid(True)

    plt.tight_layout()
    plt.show()

### Recomendación Final con Filtro de Mercado y $100,000
Une el portafolio de **Máximo Sharpe del Top 5** con el **filtro SMA 200** (decisión al último cierre de mes, como en el backtest) y calcula cuántas acciones enteras comprar con el capital.

In [ ]:
CAPITAL = 100_000
COSTO_OPERACION = 0.001   # 10 pb por dólar operado (mismo supuesto del backtest)

print("=" * 70)
print("        RECOMENDACIÓN FINAL — V5 (LAS 3 MEJORAS)")
print("=" * 70)
print(f"Tasa libre de riesgo (T-Bill): {tasa_libre_riesgo_anual:.2%}")
print(f"Filtro de mercado al cierre de mes {fin_mes.date()}: {'ALCISTA' if alcista_fin_mes else 'BAJISTA'} "
      f"| hoy: {'alcista' if mercado_alcista else 'bajista'} (SPY {distancia_sma200:+.2%} vs SMA 200)")

if alcista_fin_mes:
    precios_hoy = df_precios_unidos[top_tickers].iloc[-1]
    acciones_comprar = np.floor(CAPITAL * pesos_top5_ms.values * (1 - 2 * COSTO_OPERACION) / precios_hoy.values)
    inversion = acciones_comprar * precios_hoy.values
    plan = pd.DataFrame({
        'Acción': [t.upper() for t in top_tickers],
        'Peso Max Sharpe': pesos_top5_ms.values,
        'Precio': precios_hoy.values,
        'Acciones a comprar': acciones_comprar.astype(int),
        'Inversión': inversion,
    })
    plan = plan[plan['Acciones a comprar'] > 0]
    efectivo = CAPITAL - inversion.sum() - inversion.sum() * COSTO_OPERACION
    display(plan.style.format({'Peso Max Sharpe': '{:.2%}', 'Precio': '${:,.2f}', 'Inversión': '${:,.2f}'}).hide(axis='index'))
    print(f"Invertido: ${inversion.sum():,.2f} | Costos estimados: ${inversion.sum() * COSTO_OPERACION:,.2f} | "
          f"Efectivo sobrante: ${efectivo:,.2f}")
    print("\nReglas: rebalancear cada trimestre (último día hábil) y revisar el filtro a fin de cada mes.")
else:
    print("\nSPY está por debajo de su SMA 200: la estrategia queda 100 % en EFECTIVO / T-Bills")
    print(f"(p. ej. un ETF de T-Bills como SGOV o BIL) rindiendo ~{tasa_libre_riesgo_anual:.2%} anual.")
    print("Volver a comprar el Top 5 cuando SPY cierre un mes por encima de su SMA 200.")

# Paso 8: Backtest Walk-Forward de 20 Años con las 3 Mejoras
Simula haber invertido **$100,000** hace 20 años aplicando **cada trimestre** este mismo análisis, usando **solo los datos disponibles en cada fecha**:

1. **Universo histórico (Mejora 3):** en cada fecha solo se consideran las acciones que eran miembros del S&P 500 en ese momento (composición diaria de [github.com/fja05680/sp500](https://github.com/fja05680/sp500)).
2. Ventana de 5 años, filtro de volumen > 1M y cobertura ≥ 95 %, **Score General** y Top 5 con pesos de **Máximo Sharpe** (igual que los Pasos 1–7).
3. **Tasa libre de riesgo dinámica (Mejora 1):** el T-Bill vigente en cada fecha.
4. **Filtro SMA 200 (Mejora 2):** a fin de cada mes, si SPY < SMA 200 se pasa a efectivo.
5. Ejecución al cierre del día siguiente, acciones enteras, costos de 10 pb y el efectivo rinde el T-Bill.

> ⏱️ Opcional: tarda ~10–15 minutos (descarga ~1,000 tickers con 26 años de historia). Limitación: Yahoo Finance no tiene precios de muchas empresas que quebraron o fueron adquiridas, así que el sesgo de supervivencia se reduce pero no desaparece.

### Fase 8.1: Composición Histórica del S&P 500 y Descarga de Precios

In [ ]:
from urllib.parse import quote

ANIOS_BT = 20
VENTANA_BT = 1260          # 5 años de sesiones, igual que el análisis
TOP_N_BT = 5
VOL_MIN_BT = 1_000_000
COBERTURA_BT = 0.95
COSTO_BT = 0.001           # 10 pb por dólar operado
N_MC_BOOT = 2000           # caminos del Monte Carlo bootstrap
N_MC_AZAR = 500            # carteras aleatorias para la prueba de habilidad
SEED_BT = 42

# 1. Composición diaria histórica del S&P 500 (point-in-time)
repo = requests.get('https://api.github.com/repos/fja05680/sp500/contents/', headers=header, timeout=30).json()
nombres = [f['name'] for f in repo if isinstance(f, dict) and 'Historical Components' in f.get('name', '')]
nombre_csv = next((n for n in nombres if 'Updated' in n), nombres[0])
url_csv = 'https://raw.githubusercontent.com/fja05680/sp500/master/' + quote(nombre_csv)
composicion = pd.read_csv(StringIO(requests.get(url_csv, timeout=120).text), parse_dates=['date'])

hoy = pd.Timestamp.today().normalize()
inicio_datos = hoy - pd.DateOffset(years=ANIOS_BT + 6)          # 20 años + 5 de ventana + margen
composicion = composicion[composicion['date'] >= inicio_datos - pd.DateOffset(years=1)]
miembros_hist = {d: set(t.replace('.', '-') for t in s.split(',')) for d, s in zip(composicion['date'], composicion['tickers'])}
todos_hist = sorted(set().union(*miembros_hist.values()) | set(tickers_muestra))
print(f"Composición histórica: {len(miembros_hist)} cambios desde {composicion['date'].min().date()} | "
      f"{len(todos_hist)} tickers que alguna vez fueron miembros")

# 2. Precios ajustados de todos los miembros históricos + SPY, y la tasa del T-Bill
print("Descargando precios históricos (puede tardar varios minutos)...")
bloque_bt = yf.download(todos_hist + ['SPY'], start=inicio_datos.strftime('%Y-%m-%d'), progress=False,
                        auto_adjust=True, threads=True)
close_bt_all = bloque_bt['Close'].sort_index()
volume_bt_all = bloque_bt['Volume'].sort_index()
con_datos = close_bt_all.columns[close_bt_all.notna().any()]
close_bt_all, volume_bt_all = close_bt_all[con_datos], volume_bt_all[con_datos]
irx_bt = yf.download('^IRX', start=inicio_datos.strftime('%Y-%m-%d'), progress=False,
                     auto_adjust=True)['Close'].squeeze() / 100

spy_bt = close_bt_all['SPY']
close_bt = close_bt_all.drop(columns=['SPY'])
volume_bt = volume_bt_all.drop(columns=['SPY'])
fechas_bt = close_bt.index
irx_bt = irx_bt.reindex(fechas_bt).ffill().bfill()
rf_diario_bt = (1 + irx_bt) ** (1 / 252) - 1
sma200_bt = spy_bt.rolling(200).mean()
FIN_BT = fechas_bt[-1]
INICIO_BT = FIN_BT - pd.DateOffset(years=ANIOS_BT)

_fechas_comp = np.array(sorted(miembros_hist), dtype='datetime64[ns]')
_lista_comp = [miembros_hist[k] for k in sorted(miembros_hist)]
def miembros_en(t):
    """Tickers que pertenecían al S&P 500 en la fecha t."""
    return _lista_comp[np.searchsorted(_fechas_comp, np.datetime64(t), side='right') - 1]

sin_precio = len(todos_hist) - close_bt.shape[1]
print(f"Con precios en Yahoo: {close_bt.shape[1]} de {len(todos_hist)} ({sin_precio} sin datos: quiebras/adquisiciones)")
print(f"Backtest: {INICIO_BT.date()} a {FIN_BT.date()}")

### Fase 8.2: Motor del Backtest (Señal, Filtro y Simulación con Acciones Enteras)

In [ ]:
def max_sharpe_bt(mu, cov, rf):
    n = len(mu)
    def neg_sharpe(w):
        return -((w @ mu - rf) / np.sqrt(w @ cov @ w))
    res = sco.minimize(neg_sharpe, np.full(n, 1 / n), method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=({'type': 'eq', 'fun': lambda x: x.sum() - 1},))
    w = np.clip(res.x, 0, None)
    return w / w.sum()


def senal_bt(t):
    """Pasos 1-7 con datos hasta t: universo histórico, Score General, Top 5 y pesos Max Sharpe."""
    i = fechas_bt.get_loc(t)
    if i + 1 < VENTANA_BT:
        return None
    vent_p = close_bt.iloc[i + 1 - VENTANA_BT:i + 1]
    vent_v = volume_bt.iloc[i + 1 - VENTANA_BT:i + 1]
    rf_t = float(irx_bt.iloc[i])                                        # MEJORA 1
    cob = vent_p.notna().mean()
    en_indice = cob.index.isin(list(miembros_en(t)))                    # MEJORA 3
    filtro = (cob >= COBERTURA_BT) & (vent_v.mean() > VOL_MIN_BT) & vent_p.iloc[-1].notna() & en_indice
    elegibles = cob.index[filtro]
    p = vent_p[elegibles].dropna()
    r = p.pct_change().dropna()
    mu, vol = r.mean() * 252, r.std() * np.sqrt(252)
    df = pd.DataFrame({'Rendimiento': mu, 'Sharpe': (mu - rf_t) / vol,
                       'Control_DD': 1 + (p / p.cummax() - 1).min(), 'Baja_Vol': 1 / vol})
    nrm = (df - df.min()) / (df.max() - df.min())
    df['Score'] = 0.35 * nrm['Sharpe'] + 0.25 * nrm['Rendimiento'] + 0.25 * nrm['Control_DD'] + 0.15 * nrm['Baja_Vol']
    top = df['Score'].nlargest(TOP_N_BT).index.tolist()
    mu5, cov5 = mu[top].values, r[top].cov().values * 252
    w = max_sharpe_bt(mu5, cov5, rf_t)
    return {'fecha': t, 'top': top, 'pesos': w, 'exp_ret': w @ mu5, 'exp_vol': np.sqrt(w @ cov5 @ w),
            'score': df['Score'], 'n_univ': len(elegibles)}


def construir_eventos(senales):
    """MEJORA 2: a fin de mes, si SPY < SMA200 se pasa a efectivo; al volver arriba se recompra la última señal."""
    por_fecha = {s['fecha']: s for s in senales}
    s_f = pd.Series(fechas_bt, index=fechas_bt)
    fin_mes = set(pd.Timestamp(x) for x in s_f.resample('ME').last().dropna().values)
    calendario = sorted(d for d in fin_mes | set(por_fecha) if senales[0]['fecha'] <= d < FIN_BT)
    eventos, actual, estado = [], None, None
    for d in calendario:
        if d in por_fecha:
            actual = por_fecha[d]
        alcista = bool(spy_bt.loc[d] > sma200_bt.loc[d])
        if d in por_fecha or alcista != estado:
            if alcista:
                eventos.append(actual if d in por_fecha else {**actual, 'fecha': d})
            else:
                eventos.append({'fecha': d, 'top': [], 'pesos': np.array([])})
            estado = alcista
    return eventos


def simular_bt(eventos, capital=CAPITAL, costo=COSTO_BT):
    """Ejecuta cada evento al cierre de la sesión siguiente, con acciones enteras."""
    px = close_bt.ffill()
    exec_idx = {fechas_bt.get_loc(e['fecha']) + 1: e for e in eventos}
    inicio = min(exec_idx)
    efectivo, acciones = float(capital), pd.Series(dtype=float)
    nav, log = {}, []
    for i in range(inicio, len(fechas_bt)):
        if i > inicio:
            efectivo *= 1 + rf_diario_bt.iloc[i]
        if i in exec_idx:
            e, precios = exec_idx[i], px.iloc[i]
            valor = efectivo + (acciones * precios[acciones.index]).sum()
            objetivo = pd.Series(np.floor(valor * e['pesos'] * (1 - 2 * costo) / precios[e['top']].values), index=e['top'])
            todos = acciones.index.union(objetivo.index)
            delta = objetivo.reindex(todos, fill_value=0) - acciones.reindex(todos, fill_value=0)
            monto = (delta.abs() * precios[todos]).sum()
            efectivo -= (delta * precios[todos]).sum() + monto * costo
            antes = acciones.to_dict()
            acciones = objetivo[objetivo > 0]
            log.append({'senal': e['fecha'], 'ejecucion': fechas_bt[i], 'valor': valor, 'antes': antes,
                        'despues': acciones.to_dict(), 'precios': precios[todos].to_dict()})
        nav[fechas_bt[i]] = efectivo + (acciones * px.iloc[i][acciones.index]).sum()
    return pd.Series(nav), log


def metricas_bt(nav, bench=None):
    r = nav.pct_change().dropna()
    ex = r - rf_diario_bt.reindex(r.index).fillna(0)
    anios = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / anios) - 1
    dd = nav / nav.cummax() - 1
    neg = ex[ex < 0]
    anual = pd.concat([pd.Series([nav.iloc[0]], index=[nav.index[0] - pd.Timedelta(days=1)]),
                       nav.resample('YE').last()]).pct_change().dropna()
    m = {'Valor final': nav.iloc[-1], 'CAGR': cagr, 'Volatilidad': r.std() * np.sqrt(252),
         'Sharpe': ex.mean() / ex.std() * np.sqrt(252),
         'Sortino': ex.mean() * 252 / (np.sqrt((neg ** 2).sum() / len(ex)) * np.sqrt(252)),
         'Max Drawdown': dd.min(), 'Calmar': cagr / abs(dd.min()),
         'VaR 95% diario': np.percentile(r, 5), 'Mejor año': anual.max(), 'Peor año': anual.min()}
    if bench is not None:
        b = bench.reindex(nav.index).pct_change().dropna().reindex(r.index)
        bx = b - rf_diario_bt.reindex(r.index).fillna(0)
        beta = np.cov(ex, bx)[0, 1] / bx.var()
        m.update({'Beta vs SPY': beta, 'Alfa anual': (ex.mean() - beta * bx.mean()) * 252,
                  'Correlación vs SPY': r.corr(b)})
    return m

print("Motor del backtest listo.")

### Fase 8.3: Ejecución y Resultados con $100,000

In [ ]:
s_f = pd.Series(fechas_bt, index=fechas_bt)
fechas_reb = [pd.Timestamp(x) for x in s_f.resample('QE').last().dropna().values]
fechas_reb = [d for d in fechas_reb if INICIO_BT - pd.Timedelta(days=1) <= d < FIN_BT]

print(f"Calculando {len(fechas_reb)} señales trimestrales...")
senales_bt = [s for s in (senal_bt(d) for d in fechas_reb) if s is not None]
eventos_bt = construir_eventos(senales_bt)
nav_bt, log_bt = simular_bt(eventos_bt)
inicio_nav = nav_bt.index[0]

# Referencias: SPY y SPY con el mismo filtro SMA 200
spy_nav = CAPITAL * spy_bt.loc[inicio_nav:] / spy_bt.loc[inicio_nav]
posicion = pd.Series(np.nan, index=fechas_bt)
for d in [pd.Timestamp(x) for x in s_f.resample('ME').last().dropna().values]:
    if senales_bt[0]['fecha'] <= d < FIN_BT:
        posicion.iloc[fechas_bt.get_loc(d) + 1] = float(spy_bt.loc[d] > sma200_bt.loc[d])
posicion = posicion.ffill().loc[inicio_nav:]
r_filtro = np.where(posicion.shift(1) == 1, spy_bt.pct_change().loc[inicio_nav:], rf_diario_bt.loc[inicio_nav:])
r_filtro = r_filtro - posicion.diff().abs().fillna(0).values * COSTO_BT
r_filtro[0] = 0
spy_filtro_nav = pd.Series(CAPITAL * np.cumprod(1 + r_filtro), index=posicion.index)

tabla_bt = pd.DataFrame({
    'Estrategia V5 (las 3 mejoras)': metricas_bt(nav_bt, spy_bt),
    'SPY': metricas_bt(spy_nav, spy_bt),
    'SPY con filtro SMA 200': metricas_bt(spy_filtro_nav, spy_bt),
})
formatos = {'Valor final': '${:,.0f}', 'Sharpe': '{:.2f}', 'Sortino': '{:.2f}', 'Calmar': '{:.2f}',
            'Beta vs SPY': '{:.2f}', 'Correlación vs SPY': '{:.2f}'}
display(tabla_bt.apply(lambda col: [formatos.get(k, '{:.2%}').format(v) for k, v in col.items()]))

en_efectivo = sum(1 for e in eventos_bt if not e['top'])
print(f"Rebalanceos trimestrales: {len(senales_bt)} | pasos a efectivo por el filtro: {en_efectivo}")

fig, ax = plt.subplots(2, 1, figsize=(12, 9), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
for nombre, serie, estilo in [('Estrategia V5', nav_bt, '-'), ('SPY', spy_nav, '-'), ('SPY con filtro SMA 200', spy_filtro_nav, '--')]:
    ax[0].plot(serie.index, serie, estilo, label=nombre, lw=2 if 'V5' in nombre else 1)
ax[0].set_yscale('log'); ax[0].legend(); ax[0].grid(True, alpha=.3)
ax[0].set_title(f'$100,000 — Backtest walk-forward V5 ({INICIO_BT.year}–{FIN_BT.year})')
for nombre, serie in [('Estrategia V5', nav_bt), ('SPY', spy_nav)]:
    ax[1].plot(serie.index, serie / serie.cummax() - 1, label=nombre)
ax[1].set_ylabel('Drawdown'); ax[1].legend(); ax[1].grid(True, alpha=.3)
plt.tight_layout(); plt.show()

anuales_bt = pd.DataFrame({k: pd.concat([pd.Series([v.iloc[0]], index=[v.index[0] - pd.Timedelta(days=1)]),
                                          v.resample('YE').last()]).pct_change().dropna()
                           for k, v in {'Estrategia V5': nav_bt, 'SPY': spy_nav, 'SPY con filtro': spy_filtro_nav}.items()})
anuales_bt.index = anuales_bt.index.year
print("Retorno por año:")
display(anuales_bt.style.format('{:.2%}'))

### Fase 8.4: ¿Qué Tan Preciso Fue el Modelo con el Tiempo?
- **Retorno prometido vs real:** el retorno esperado del optimizador contra lo que realmente pasó el trimestre siguiente.
- **IC del Score:** correlación de rangos (Spearman) entre el Score General y el retorno futuro de todo el universo. Si es ≈ 0, el Score no predice qué acciones subirán.

In [ ]:
from scipy.stats import spearmanr

filas_prec = []
for k, s in enumerate(senales_bt):
    a = fechas_bt.get_loc(s['fecha']) + 1
    b = fechas_bt.get_loc(senales_bt[k + 1]['fecha']) + 1 if k + 1 < len(senales_bt) else len(fechas_bt) - 1
    if b - a < 10:
        continue
    tramo = nav_bt.loc[fechas_bt[a]:fechas_bt[b]]
    r_real = tramo.iloc[-1] / tramo.iloc[0] - 1
    fwd = (close_bt.iloc[b] / close_bt.iloc[a] - 1).reindex(s['score'].index).dropna()
    filas_prec.append({'fecha': s['fecha'].date(), 'top': ', '.join(s['top']), 'ret_esperado': s['exp_ret'],
                       'ret_real_anual': (1 + r_real) ** (252 / (len(tramo) - 1)) - 1,
                       'vence_spy': r_real > spy_bt.iloc[b] / spy_bt.iloc[a] - 1,
                       'ic': spearmanr(s['score'][fwd.index], fwd).statistic})
prec_bt = pd.DataFrame(filas_prec)

print(f"Retorno anual prometido (promedio): {prec_bt['ret_esperado'].mean():.2%}")
print(f"Retorno anual real (mediana):       {prec_bt['ret_real_anual'].median():.2%}")
print(f"Correlación prometido vs real:      {prec_bt[['ret_esperado', 'ret_real_anual']].corr().iloc[0, 1]:.3f}")
print(f"IC promedio del Score:              {prec_bt['ic'].mean():.3f} "
      f"(t = {prec_bt['ic'].mean() / prec_bt['ic'].std() * np.sqrt(len(prec_bt)):.2f})")
print(f"Trimestres que le ganó a SPY:       {prec_bt['vence_spy'].mean():.0%}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(prec_bt['ret_esperado'], prec_bt['ret_real_anual'].clip(-1, 2.5), alpha=.6)
lim = [0, prec_bt['ret_esperado'].max()]
ax[0].plot(lim, lim, 'r--', label='Predicción perfecta'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_xlabel('Retorno prometido (anual)'); ax[0].set_ylabel('Retorno real (anualizado)')
ax[0].set_title('Precisión del retorno esperado')
ax[1].bar(pd.to_datetime(prec_bt['fecha']), prec_bt['ic'], width=60,
          color=np.where(prec_bt['ic'] >= 0, 'seagreen', 'indianred'))
ax[1].axhline(prec_bt['ic'].mean(), color='k', ls='--', label=f"IC medio {prec_bt['ic'].mean():.3f}")
ax[1].set_title('IC del Score General por trimestre'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### Fase 8.5: Monte Carlo
1. **Bootstrap pareado:** remuestrea bloques de 21 días de los retornos diarios reales (los mismos bloques para la estrategia y SPY) para simular futuros de 20 años con $100,000.
2. **¿Habilidad o suerte?:** compara la estrategia contra carteras de 5 acciones **elegidas al azar** del mismo universo histórico, en las mismas fechas y con el mismo filtro de mercado.

In [ ]:
rng_bt = np.random.default_rng(SEED_BT)

# 1. Bootstrap pareado por bloques
def mc_bootstrap(serie, n_paths, horizonte=252 * ANIOS_BT, bloque=21, seed=SEED_BT + 1):
    r = serie.pct_change().dropna().values
    rng_local = np.random.default_rng(seed)        # misma semilla => mismos bloques en cada serie
    finales, mdds, muestras = np.empty(n_paths), np.empty(n_paths), []
    for p in range(n_paths):
        idx = rng_local.integers(0, len(r) - bloque, size=horizonte // bloque + 1)
        v = CAPITAL * np.cumprod(1 + np.concatenate([r[j:j + bloque] for j in idx])[:horizonte])
        finales[p], mdds[p] = v[-1], (v / np.maximum.accumulate(v) - 1).min()
        muestras.append(v[::21])
    return finales, mdds, np.percentile(np.array(muestras), [5, 25, 50, 75, 95], axis=0)

fin_e, mdd_e, band_e = mc_bootstrap(nav_bt, N_MC_BOOT)
fin_s, mdd_s, band_s = mc_bootstrap(spy_nav, N_MC_BOOT)
resumen_mc = pd.DataFrame({
    'Estrategia V5': [np.percentile(fin_e, 5), np.median(fin_e), np.percentile(fin_e, 95), np.median(mdd_e),
                      (mdd_e < -0.5).mean(), (fin_e < CAPITAL).mean()],
    'SPY': [np.percentile(fin_s, 5), np.median(fin_s), np.percentile(fin_s, 95), np.median(mdd_s),
            (mdd_s < -0.5).mean(), (fin_s < CAPITAL).mean()],
}, index=['Valor final p5', 'Valor final mediana', 'Valor final p95', 'Caída máx. mediana',
          'Prob. caída > 50 %', 'Prob. perder capital'])
display(resumen_mc.apply(lambda c: [f'${v:,.0f}' if i < 3 else f'{v:.1%}' for i, v in enumerate(c)]))
print(f"Probabilidad de que la estrategia termine arriba de SPY en 20 años: {(fin_e > fin_s).mean():.0%}")

# 2. Carteras aleatorias con las mismas reglas (motor vectorizado, pesos que derivan entre rebalanceos)
R_bt, RF_bt = close_bt.pct_change(fill_method=None).fillna(0).values, rf_diario_bt.values
col_idx = {c: j for j, c in enumerate(close_bt.columns)}
idx_e = [fechas_bt.get_loc(e['fecha']) for e in eventos_bt]
univ_e = [[col_idx[c] for c in e['score'].index] if e['top'] else [] for e in eventos_bt]

def backtest_pesos(pesos, columnas, costo=COSTO_BT):
    n, inicio = len(fechas_bt), idx_e[0] + 1
    nav = np.empty(n - inicio); nav[0] = v = 1.0; previo = {}
    for k, i0 in enumerate(idx_e):
        a = i0 + 1
        b = idx_e[k + 1] + 1 if k + 1 < len(idx_e) else n - 1
        cols, w = list(columnas[k]), np.asarray(pesos[k], dtype=float)
        nuevo = dict(zip(cols, w))
        v *= 1 - costo * sum(abs(nuevo.get(c, 0) - previo.get(c, 0)) for c in set(nuevo) | set(previo))
        nav[a - inicio] = v
        if b > a and not cols:
            nav[a + 1 - inicio:b + 1 - inicio] = v * np.cumprod(1 + RF_bt[a + 1:b + 1]); v = nav[b - inicio]; previo = {}
        elif b > a:
            crec = np.cumprod(1 + R_bt[a + 1:b + 1][:, cols], axis=0)
            nav[a + 1 - inicio:b + 1 - inicio] = v * (crec @ w); v = nav[b - inicio]
            wf = w * crec[-1]; previo = dict(zip(cols, wf / wf.sum()))
        else:
            previo = nuevo
    return pd.Series(nav, index=fechas_bt[inicio:])

def cagr_de(nv):
    return nv.iloc[-1] ** (365.25 / (nv.index[-1] - nv.index[0]).days) - 1

cagr_estr = cagr_de(backtest_pesos([e['pesos'] for e in eventos_bt], [[col_idx[c] for c in e['top']] for e in eventos_bt]))
cagr_azar = []
for _ in range(N_MC_AZAR):
    cols_k = [list(rng_bt.choice(c, TOP_N_BT, replace=False)) if c else [] for c in univ_e]
    pesos_k = [rng_bt.dirichlet(np.ones(TOP_N_BT)) if c else np.array([]) for c in univ_e]
    cagr_azar.append(cagr_de(backtest_pesos(pesos_k, cols_k)))
cagr_azar = np.array(cagr_azar)
percentil = (cagr_azar < cagr_estr).mean()
print(f"CAGR estrategia: {cagr_estr:.2%} | azar: mediana {np.median(cagr_azar):.2%} "
      f"(p5 {np.percentile(cagr_azar, 5):.2%}, p95 {np.percentile(cagr_azar, 95):.2%})")
print(f"Percentil de la estrategia: {percentil:.0%} | p-valor: {1 - percentil:.3f} "
      f"({'significativo' if 1 - percentil < 0.05 else 'no se distingue del azar'} al 5 %)")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
meses = np.arange(band_e.shape[1]) / 12
for nombre, band, col in [('Estrategia V5', band_e, 'C0'), ('SPY', band_s, 'C3')]:
    ax[0].fill_between(meses, band[0], band[4], color=col, alpha=.12)
    ax[0].fill_between(meses, band[1], band[3], color=col, alpha=.25)
    ax[0].plot(meses, band[2], color=col, label=f'{nombre} (mediana, 50 % y 90 %)')
ax[0].set_yscale('log'); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_xlabel('Años')
ax[0].set_title(f'Monte Carlo bootstrap – {N_MC_BOOT} caminos de {ANIOS_BT} años')
ax[1].hist(cagr_azar, bins=30, alpha=.7, label='Carteras aleatorias')
ax[1].axvline(cagr_estr, color='r', lw=2, label=f'Estrategia ({cagr_estr:.1%})')
ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title('Estrategia vs selección al azar')
plt.tight_layout(); plt.show()

### Fase 8.6: Todas las Operaciones y Acciones Compradas en los 20 Años

In [ ]:
filas_ops = []
for l in log_bt:
    for t in sorted(set(l['antes']) | set(l['despues'])):
        a0, a1 = l['antes'].get(t, 0), l['despues'].get(t, 0)
        delta, precio = a1 - a0, l['precios'][t]
        tipo = 'Compra' if a0 == 0 else 'Venta total' if a1 == 0 else 'Aumenta' if delta > 0 else 'Reduce' if delta < 0 else 'Mantiene'
        filas_ops.append({'Ejecución': l['ejecucion'].date(), 'Acción': t, 'Operación': tipo, 'Acciones antes': int(a0),
                          'Acciones después': int(a1), 'Precio': precio, 'Monto operado': abs(delta) * precio,
                          'Peso': a1 * precio / l['valor'], 'Valor portafolio': l['valor']})
    if not l['despues']:
        filas_ops.append({'Ejecución': l['ejecucion'].date(), 'Acción': 'EFECTIVO', 'Operación': 'A efectivo (SPY < SMA200)',
                          'Acciones antes': 0, 'Acciones después': 0, 'Precio': np.nan, 'Monto operado': 0.0,
                          'Peso': 1.0, 'Valor portafolio': l['valor']})
operaciones_bt = pd.DataFrame(filas_ops)

tenencias = operaciones_bt[(operaciones_bt['Acciones después'] > 0)]
acciones_bt = (tenencias.groupby('Acción')
               .agg(Trimestres_en_cartera=('Ejecución', 'count'), Primera_compra=('Ejecución', 'min'),
                    Ultima_vez=('Ejecución', 'max'), Peso_promedio=('Peso', 'mean'))
               .join(operaciones_bt[operaciones_bt['Operación'] == 'Compra'].groupby('Acción').size().rename('Veces_comprada'))
               .join(operaciones_bt[operaciones_bt['Monto operado'] > 0]
                     .loc[lambda d: d['Acciones después'] > d['Acciones antes']].groupby('Acción')['Monto operado'].sum()
                     .rename('Monto_comprado_total'))
               .sort_values('Trimestres_en_cartera', ascending=False))

print(f"{len(operaciones_bt)} operaciones | {len(acciones_bt)} acciones distintas compradas en {ANIOS_BT} años")
print(', '.join(f'{t} ({n})' for t, n in acciones_bt['Trimestres_en_cartera'].items()))
display(acciones_bt.style.format({'Peso_promedio': '{:.1%}', 'Monto_comprado_total': '${:,.0f}'}))
print("\nÚltimas 25 operaciones:")
display(operaciones_bt.tail(25).style.format({'Precio': '${:,.2f}', 'Monto operado': '${:,.0f}', 'Peso': '{:.1%}',
                                               'Valor portafolio': '${:,.0f}'}).hide(axis='index'))

operaciones_bt.to_csv('operaciones_v5_backtest.csv', index=False)
acciones_bt.to_csv('acciones_v5_backtest.csv')
print("Guardado: operaciones_v5_backtest.csv y acciones_v5_backtest.csv")
try:
    from google.colab import files   # en Colab, descarga los CSV
    files.download('operaciones_v5_backtest.csv')
    files.download('acciones_v5_backtest.csv')
except ImportError:
    pass